# Embedding Model Benchmark — EU Legislative Retrieval

**Research question**: Which sentence-transformer best recovers the correct proposal article
when given a lobbying/stakeholder text that has had its explicit article citations removed?

## Design

Post-proposal HYS feedback submissions often cite articles directly
(e.g. *"Article 5 should require..."*). These citations are self-labelling ground truth.

| Step | What happens |
|------|--------------|
| 1 | Load post-proposal feedback chunks that contain explicit `Article N` / `Recital N` citations |
| 2 | Strip those citations → decontextualized feedback |
| 3 | Strip article headers (number + title) from proposal articles → clean article body |
| 4 | Embed both with 4 models; rank retrieval of correct article |
| 5 | Flag near-paraphrase pairs (ROUGE-L) to separate lexical from semantic signal |

### Three evaluation variants

| Variant | Query | Corpus | Meaning |
|---------|-------|--------|---------|
| **baseline** | feedback *with* citations | articles *with* header | Upper bound — identifiers present on both sides |
| **actual** | feedback *without* citations | articles *with* header | **Real pipeline proxy** — lobbying stmt vs. article |
| **strict** | feedback *without* citations | articles *without* header | Semantic lower bound |

The **actual** variant is the most important: it mirrors the real influence-analysis task where
pre-proposal lobbying statements have no article references but articles still carry their header.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
PROCEDURE_ID = '2021/0106(COD)'

MODELS = [
    {
        'name':           'mpnet-base-v2',
        'hf_id':          'sentence-transformers/all-mpnet-base-v2',
        'query_prefix':   '',
        'passage_prefix': '',
    },
    {
        'name':           'bge-large-en-v1.5',
        'hf_id':          'BAAI/bge-large-en-v1.5',
        'query_prefix':   'Represent this sentence for searching relevant passages: ',
        'passage_prefix': '',
    },
    {
        'name':           'e5-large-v2',
        'hf_id':          'intfloat/e5-large-v2',
        'query_prefix':   'query: ',
        'passage_prefix': 'passage: ',
    },
    {
        # Direct apples-to-apples baseline for the instruct variant below.
        # Handles non-English submissions (French, German etc.) without degrading.
        'name':           'multilingual-e5-large',
        'hf_id':          'intfloat/multilingual-e5-large',
        'query_prefix':   'query: ',
        'passage_prefix': 'passage: ',
    },
    {
        # Instruct variant of multilingual-e5-large.
        # Queries take a full task instruction; passages are encoded with no prefix.
        'name':           'multilingual-e5-large-instruct',
        'hf_id':          'intfloat/multilingual-e5-large-instruct',
        'query_prefix':   (
            'Instruct: Given a stakeholder lobbying position on EU legislation, '
            'retrieve the most relevant legislative article or recital\nQuery: '
        ),
        'passage_prefix': '',
    },
    {
        # Legal sentence-BERT trained on EUR-Lex and multilingual legal corpora.
        'name':           'sbert-legal-xlm-roberta',
        'hf_id':          'Stern5497/sbert-legal-xlm-roberta-base',
        'query_prefix':   '',
        'passage_prefix': '',
    },
]

K_VALUES             = [1, 3, 5, 10]
MIN_CLEANED_LEN      = 80
PARAPHRASE_THRESHOLD = 0.30
MIN_GOLD_PAIRS       = 20

In [ ]:
import os, re, gc, json
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from dotenv import load_dotenv
from supabase import create_client
from sentence_transformers import SentenceTransformer

load_dotenv()
supabase = create_client(
    os.getenv('SUPABASE_URL'),
    os.getenv('SUPABASE_SERVICE_ROLE_KEY'),
)
Path('results').mkdir(exist_ok=True)
print('Connected to Supabase.')

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def paginate(table, select, filter_fn=None, page_size=1000):
    rows, offset = [], 0
    while True:
        q = supabase.table(table).select(select)
        if filter_fn:
            q = filter_fn(q)
        page = q.range(offset, offset + page_size - 1).execute().data or []
        rows.extend(page)
        if len(page) < page_size:
            break
        offset += page_size
    return rows


def embed_texts(model, texts, prefix='', batch_size=32):
    """Encode texts with optional prefix; returns L2-normalised (n, dim) float32 array."""
    if not texts:
        return np.empty((0, model.get_sentence_embedding_dimension()), dtype=np.float32)
    prefixed = [prefix + t for t in texts] if prefix else texts
    return model.encode(
        prefixed,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )


print('Helpers ready.')

## 1. Data Loading

In [ ]:
proc = (
    supabase.table('procedures')
    .select('title, proposal_date, commission_document')
    .eq('id', PROCEDURE_ID)
    .single()
    .execute()
    .data
)
PROPOSAL_DATE = proc['proposal_date']
print(f'Procedure:     {PROCEDURE_ID}')
print(f'Title:         {proc["title"]}')
print(f'Proposal date: {PROPOSAL_DATE}')

In [ ]:
# Load structured proposal articles (document_version = 'proposal')
art_rows = paginate(
    'procedure_articles',
    'element_type, element_number, title, content, sort_order',
    lambda q: (
        q.eq('procedure_id', PROCEDURE_ID)
         .eq('document_version', 'proposal')
         .order('sort_order')
    ),
)

if not art_rows:
    raise ValueError(
        f'No proposal articles in procedure_articles for {PROCEDURE_ID}.\n'
        'Populate via the diamond asset, or switch PROCEDURE_ID.'
    )

n_rec = sum(1 for a in art_rows if a['element_type'] == 'recital')
n_art = sum(1 for a in art_rows if a['element_type'] == 'article')
print(f'Proposal elements: {n_rec} recitals + {n_art} articles = {len(art_rows)} total')

# Canonical lookup: (element_type, normalised_number) -> row
article_lookup = {
    (a['element_type'], a['element_number'].lstrip('0') or '0'): a
    for a in art_rows
}

In [ ]:
fb_rows = paginate(
    'hys_feedback_chunks',
    'id, feedback_id, chunk_index, chunk_text, organisation_name, transparency_reg_id, date_feedback',
    lambda q: (
        q.eq('procedure_id', PROCEDURE_ID)
         .gte('date_feedback', PROPOSAL_DATE)
    ),
)
print(f'Post-proposal feedback chunks: {len(fb_rows)}')
print(f'Unique organisations:          {len({r.get("organisation_name") for r in fb_rows})}')

## 2. Gold Standard Construction

A **gold pair** is a `(feedback_chunk, article)` tuple where the chunk explicitly names
the article by number.  We create three text variants:

- **feedback_raw** — original chunk, citations intact
- **feedback_clean** — chunk with all `Article N` / `Recital N` references removed
- **art_raw** — `"article N: Title\n<body>"` (header + body)
- **art_clean** — body only (no number, no title)

Combining these gives us the three evaluation variants (baseline / actual / strict).

In [ ]:
# ── Citation regex ────────────────────────────────────────────────────────────
_NUM  = r'\(?\d+[a-z]?\)?'
_SEP  = r'(?:\s*(?:,|and|to|-)\s*' + _NUM + r')*'
_ART  = r'(?:articles?|art\.?)'
_REC  = r'(?:recitals?)'

STRIP_RE = re.compile(
    _ART + r'\s*' + _NUM + r'(?:,\s*paragraph\s*\d+[a-z]?)?' + _SEP
    + r'|' +
    _REC + r'\s*' + _NUM + _SEP,
    re.IGNORECASE,
)

_CAP_ART = re.compile(_ART + r'\s*\(?(\d+[a-z]?)\)?', re.IGNORECASE)
_CAP_REC = re.compile(_REC + r'\s*\(?(\d+)\)?',       re.IGNORECASE)


def clean_citations(text):
    """Remove explicit article/recital references; collapse resulting whitespace."""
    out = STRIP_RE.sub('', text)
    out = re.sub(r'[ \t]{2,}', ' ', out)
    out = re.sub(r'\n{3,}', '\n\n', out)
    return out.strip()


def citation_window(text, match, window=900):
    """Return up to 2*window chars centered on a regex match.

    Gives the model the argument surrounding the citation rather than
    whatever happens to fall within a fixed chunk boundary.
    ~1800 chars fits comfortably within the 512-token limit of all four models.
    """
    start = max(0, match.start() - window)
    end   = min(len(text), match.end() + window)
    return text[start:end]


# Sanity check
_t = 'In Article 5 and recital (12), the proposal requires that Art. 3a defines the scope.'
_m = _CAP_ART.search(_t)
print('Input:   ', _t)
print('Window:  ', citation_window(_t, _m, window=20))
print('Cleaned: ', clean_citations(citation_window(_t, _m, window=20)))

In [ ]:
# ── Chunk lookup for context windows ─────────────────────────────────────────
# Keyed by (feedback_id, chunk_index) so we can grab the next chunk cheaply.
chunk_lookup = {
    (r['feedback_id'], r['chunk_index']): r['chunk_text']
    for r in fb_rows
}

# ── Build gold pairs ──────────────────────────────────────────────────────────
gold_pairs = []
skipped    = defaultdict(int)
seen_keys  = set()  # deduplicate (chunk_id, element) in case of repeated citations

for row in fb_rows:
    current = (row.get('chunk_text') or '').strip()
    if not current:
        skipped['empty_chunk'] += 1
        continue

    # Extend into the next chunk so citations at chunk boundaries have context
    nxt      = chunk_lookup.get((row['feedback_id'], row['chunk_index'] + 1), '')
    combined = (current + ' ' + nxt).strip() if nxt else current

    # Find all citation matches with their positions
    matches = (
        [('article', m) for m in _CAP_ART.finditer(combined)] +
        [('recital', m) for m in _CAP_REC.finditer(combined)]
    )
    if not matches:
        skipped['no_citation'] += 1
        continue

    for etype, match in matches:
        enum = match.group(1).lower().lstrip('0') or '0'

        if (etype, enum) not in article_lookup:
            skipped['article_not_found'] += 1
            continue

        # Deduplicate: same chunk + same article shouldn't produce two pairs
        key = (row['id'], etype, enum)
        if key in seen_keys:
            continue
        seen_keys.add(key)

        # Extract ~1800 chars centered on the citation, then strip it
        window_text = citation_window(combined, match, window=900)
        cleaned     = clean_citations(window_text)

        if len(cleaned) < MIN_CLEANED_LEN:
            skipped['too_short_after_clean'] += 1
            continue

        art = article_lookup[(etype, enum)]
        gold_pairs.append({
            'chunk_id':       row['id'],
            'org':            row.get('organisation_name'),
            'feedback_raw':   window_text,
            'feedback_clean': cleaned,
            'element_type':   etype,
            'element_number': enum,
            'art_content':    art['content'],
            'art_title':      art.get('title'),
        })

print(f'Gold pairs:              {len(gold_pairs)}')
print(f'Unique articles cited:   {len({(p["element_type"], p["element_number"]) for p in gold_pairs})}')
print(f'Unique feedback chunks:  {len({p["chunk_id"] for p in gold_pairs})}')
print(f'Skipped: {dict(skipped)}')

if len(gold_pairs) < MIN_GOLD_PAIRS:
    raise ValueError(
        f'Only {len(gold_pairs)} gold pairs (need >= {MIN_GOLD_PAIRS}).\n'
        'Try a procedure with more post-proposal feedback, e.g. 2021/0106(COD).'
    )

gold_df = pd.DataFrame(gold_pairs)

In [ ]:
# ── Article corpus arrays (built once — independent of gold pair filtering) ───
def art_header(a):
    h = f"{a['element_type']} {a['element_number']}"
    return h + (f": {a['title']}" if a.get('title') else '')

art_raw   = [art_header(a) + '\n' + a['content'] for a in art_rows]
art_clean = [a['content'] for a in art_rows]

article_keys   = [(a['element_type'], a['element_number'].lstrip('0') or '0') for a in art_rows]
art_key_to_idx = {k: i for i, k in enumerate(article_keys)}

print(f'Article corpus: {len(art_raw)} elements')
print(f'art_raw[0]:   {art_raw[0][:110]!r}')
print(f'art_clean[0]: {art_clean[0][:110]!r}')

In [ ]:
# ── Citation density filter ───────────────────────────────────────────────────
# Chunks with 5+ citations are boilerplate lists ("Articles 1, 2, 3...")
# rather than substantive per-article comments. Drop before computing ROUGE-L.
from collections import Counter

MAX_CITATIONS_PER_CHUNK = 4
cites_per_chunk = Counter(p['chunk_id'] for p in gold_pairs)
noisy_chunks    = {cid for cid, n in cites_per_chunk.items() if n >= 5}
before          = len(gold_pairs)
gold_pairs      = [p for p in gold_pairs if p['chunk_id'] not in noisy_chunks]
print(f'Density filter: dropped {before - len(gold_pairs)} pairs from '
      f'{len(noisy_chunks)} high-density chunks → {len(gold_pairs)} remaining')
print(f'Citations/chunk — mean: {sum(cites_per_chunk.values())/len(cites_per_chunk):.1f}  '
      f'median: {sorted(cites_per_chunk.values())[len(cites_per_chunk)//2]}  '
      f'max: {max(cites_per_chunk.values())}')

# ── Paraphrase detection via ROUGE-L ─────────────────────────────────────────
def _lcs_len(a, b):
    a, b = a[:150], b[:150]
    m, n = len(a), len(b)
    if not m or not n:
        return 0
    prev = [0] * (n + 1)
    for i in range(m):
        curr = [0] * (n + 1)
        for j in range(n):
            curr[j + 1] = prev[j] + 1 if a[i] == b[j] else max(prev[j + 1], curr[j])
        prev = curr
    return prev[n]


def rouge_l_f1(hyp, ref):
    h = hyp.lower().split()
    r = ref.lower().split()
    if not h or not r:
        return 0.0
    lcs  = _lcs_len(h, r)
    prec = lcs / len(h)
    rec  = lcs / len(r)
    return 2 * prec * rec / (prec + rec) if prec + rec else 0.0


gold_df = pd.DataFrame(gold_pairs)
gold_df['rouge_l']       = [rouge_l_f1(p['feedback_clean'], p['art_content']) for p in gold_pairs]
gold_df['is_paraphrase'] = gold_df['rouge_l'] >= PARAPHRASE_THRESHOLD

print(f'\nNear-paraphrases (ROUGE-L >= {PARAPHRASE_THRESHOLD}): '
      f'{gold_df["is_paraphrase"].mean():.1%}  '
      f'({gold_df["is_paraphrase"].sum()} / {len(gold_df)})')
print(f'ROUGE-L  mean={gold_df["rouge_l"].mean():.3f}  '
      f'median={gold_df["rouge_l"].median():.3f}  '
      f'max={gold_df["rouge_l"].max():.3f}')

In [ ]:
# ── Dataset summary ───────────────────────────────────────────────────────────
print('=== Gold Standard ===')
print(f'Procedure:               {PROCEDURE_ID}')
print(f'Corpus size:             {len(art_rows)} proposal elements')
print(f'Gold pairs total:        {len(gold_pairs)}')
print(f'  Semantic (non-para):   {(~gold_df["is_paraphrase"]).sum()}')
print(f'  Near-paraphrase:       {gold_df["is_paraphrase"].sum()}')
unique_cited = gold_df[['element_type', 'element_number']].drop_duplicates()
print(f'  Unique articles cited: {len(unique_cited)} / {len(art_rows)} '
      f'({len(unique_cited)/len(art_rows):.0%} coverage)')
print()
print('Top 15 most-cited elements:')
cit_dist = gold_df.groupby(['element_type', 'element_number']).size().sort_values(ascending=False)
print(cit_dist.head(15).to_string())

In [ ]:
# ── Query arrays — built from filtered gold_pairs ────────────────────────────
# Must come after density filter and ROUGE-L so indices stay aligned with gold_df.
feedback_raw   = [p['feedback_raw']   for p in gold_pairs]
feedback_clean = [p['feedback_clean'] for p in gold_pairs]
gold_art_idxs  = [art_key_to_idx[(p['element_type'], p['element_number'])] for p in gold_pairs]

print(f'Query set:    {len(feedback_clean)} gold pairs')
print(f'gold_df rows: {len(gold_df)}  (must match above)')

## 3. Embedding Evaluation

For each model we embed all four text pools once, compute three similarity matrices
(one per variant), and record Recall@k and MRR.  Models are freed from memory after
each run to avoid OOM on CPU.

In [ ]:

# Variant: (name, query_pool_key, corpus_pool_key, description)
VARIANTS = [
    ('actual', 'fb_clean', 'art_raw',   'Clean query + raw article — real pipeline proxy'),
    ('strict', 'fb_clean', 'art_clean', 'Both stripped — semantic lower bound'),
]

all_results  = []  # one dict per (model, variant)
per_query_db = {}  # (model, variant) -> list of per-query dicts
sim_db       = {}  # (model, variant) -> (n_queries, n_articles) sim matrix

for cfg in MODELS:
    name, hf_id = cfg['name'], cfg['hf_id']
    qpfx, ppfx  = cfg['query_prefix'], cfg['passage_prefix']

    print(f'\n{"="*62}')
    print(f'  {name}  ({hf_id})')
    print(f'{"="*62}')

    mdl = SentenceTransformer(hf_id)

    print('  [1/3] article corpus (raw)  ...')
    emb_art_raw   = embed_texts(mdl, art_raw,        prefix=ppfx)
    print('  [2/3] article corpus (clean)...')
    emb_art_clean = embed_texts(mdl, art_clean,      prefix=ppfx)
    print('  [3/3] feedback queries (clean)...')
    emb_fb_clean  = embed_texts(mdl, feedback_clean, prefix=qpfx)

    del mdl; gc.collect()

    pool = {
        'art_raw':   emb_art_raw,
        'art_clean': emb_art_clean,
        'fb_clean':  emb_fb_clean,
    }

    for v_name, q_key, a_key, _ in VARIANTS:
        sim = pool[q_key] @ pool[a_key].T   # (n_queries, n_articles)
        sim_db[(name, v_name)] = sim        # save for reciprocal analysis

        per_q, rr = [], []

        for qi, true_idx in enumerate(gold_art_idxs):
            scores = sim[qi]
            ranked = np.argsort(scores)[::-1]
            rank   = int(np.where(ranked == true_idx)[0][0]) + 1

            rr.append(1.0 / rank)
            per_q.append({
                'rank':          rank,
                'score_correct': float(scores[true_idx]),
                'score_top1':    float(scores[ranked[0]]),
                'top1_art_idx':  int(ranked[0]),
                'rouge_l':       gold_df.iloc[qi]['rouge_l'],
                'is_paraphrase': bool(gold_df.iloc[qi]['is_paraphrase']),
            })

        per_query_db[(name, v_name)] = per_q

        row = {'model': name, 'variant': v_name, 'mrr': float(np.mean(rr))}
        for k in K_VALUES:
            row[f'R@{k}'] = sum(1 for p in per_q if p['rank'] <= k) / len(per_q)
        all_results.append(row)

        print(f'  [{v_name:10s}]  MRR={row["mrr"]:.3f}  ' +
              '  '.join(f'R@{k}={row[f"R@{k}"]:.0%}' for k in K_VALUES))

print('\nAll models evaluated.')


## 3.5  Reciprocal Matching

A **reciprocal pair** `(feedback_i, article_j)` requires both directions to agree:
- *Forward*: article j is in feedback i's top-K_R neighbours
- *Reverse*: feedback i is in article j's top-K_R neighbours (i.e., the article "selects back")

This raises the precision bar — a noisy forward hit is unlikely to survive the reverse filter.
The cost is lower recall: if the gold article doesn't reciprocate, the pair is lost.

Three metrics are reported alongside the baseline forward metrics:

| Metric | Meaning |
|--------|---------|
| **gold survival** | % of gold pairs where the correct article passes the reciprocal filter |
| **recip MRR / R@k** | MRR / R@k re-ranked within the reciprocal candidate set (non-surviving pairs contribute 0) |
| **MRR Δ** | recip MRR − forward MRR — positive = reciprocal helps, negative = it hurts |

`K_RECIP` controls strictness: smaller = fewer pairs survive but higher precision.
Values below are swept across [5, 10, 20]; the value matching `K=5` in the main benchmark
is the most directly comparable.

In [ ]:

# ── Reciprocal matching sweep ─────────────────────────────────────────────────
K_RECIP_VALUES = [5, 10, 20]

recip_results = []

for (name, v_name), sim in sim_db.items():
    n_q, n_art = sim.shape

    for k_r in K_RECIP_VALUES:
        # Forward top-K_R mask: shape (n_q, n_art)
        fwd_top  = np.argpartition(sim, -k_r, axis=1)[:, -k_r:]   # (n_q, k_r)
        fwd_mask = np.zeros((n_q, n_art), dtype=bool)
        for qi in range(n_q):
            fwd_mask[qi, fwd_top[qi]] = True

        # Reverse top-K_R mask: for each article, which queries are its top-K_R?
        # sim[:, aj] is the score of every query against article aj
        rev_top  = np.argpartition(sim, -k_r, axis=0)[-k_r:, :]   # (k_r, n_art)
        rev_mask = np.zeros((n_q, n_art), dtype=bool)
        for aj in range(n_art):
            rev_mask[rev_top[:, aj], aj] = True

        recip_mask = fwd_mask & rev_mask  # (n_q, n_art)

        fwd_ranks, recip_ranks, gold_survived = [], [], []

        for qi, true_idx in enumerate(gold_art_idxs):
            # Forward rank
            fwd_sorted = np.argsort(sim[qi])[::-1]
            fwd_rank   = int(np.where(fwd_sorted == true_idx)[0][0]) + 1
            fwd_ranks.append(fwd_rank)

            # Reciprocal rank — non-surviving gold pairs contribute rank=inf (MRR=0)
            survived = bool(recip_mask[qi, true_idx])
            gold_survived.append(survived)

            if survived:
                cands      = np.where(recip_mask[qi])[0]
                sorted_r   = cands[np.argsort(sim[qi, cands])[::-1]]
                recip_rank = int(np.where(sorted_r == true_idx)[0][0]) + 1
            else:
                recip_rank = float('inf')
            recip_ranks.append(recip_rank)

        fwd_mrr   = float(np.mean([1 / r for r in fwd_ranks]))
        recip_mrr = float(np.mean([1 / r if r != float('inf') else 0 for r in recip_ranks]))
        survival  = float(np.mean(gold_survived))

        row = {
            'model':         name,
            'variant':       v_name,
            'K_R':           k_r,
            'fwd_MRR':       fwd_mrr,
            'recip_MRR':     recip_mrr,
            'MRR_delta':     recip_mrr - fwd_mrr,
            'gold_survival': survival,
        }
        for k in [1, 3, 5, 10]:
            row[f'fwd_R@{k}']   = sum(1 for r in fwd_ranks   if r <= k) / n_q
            row[f'recip_R@{k}'] = sum(1 for r in recip_ranks if r <= k) / n_q
        recip_results.append(row)

recip_df = pd.DataFrame(recip_results)

# ── Print summary tables ──────────────────────────────────────────────────────
for v_name, _, _, _ in VARIANTS:
    print(f'\n{"="*70}')
    print(f'  RECIPROCAL MATCHING  ({v_name.upper()} variant)')
    print(f'{"="*70}')
    sub = recip_df[recip_df['variant'] == v_name].copy()
    display_cols = ['model', 'K_R', 'fwd_MRR', 'recip_MRR', 'MRR_delta',
                    'gold_survival', 'fwd_R@5', 'recip_R@5']
    disp = sub[display_cols].sort_values(['model', 'K_R'])
    for col in disp.columns:
        if col not in ('model', 'K_R'):
            disp[col] = disp[col].map('{:+.1%}'.format if col == 'MRR_delta' else '{:.1%}'.format)
    print(disp.to_string(index=False))


In [ ]:

# ── Figure: reciprocal matching — MRR delta and gold survival by K_R ─────────
# Focus on the actual variant and the best model identified earlier.

actual_recip = recip_df[recip_df['variant'] == 'actual'].copy()
model_order_recip = (
    actual_recip[actual_recip['K_R'] == 5]
    .sort_values('fwd_MRR', ascending=False)['model']
    .tolist()
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Reciprocal Matching — actual variant', fontsize=13, fontweight='bold')

colors = ['#4878CF', '#6ACC65', '#D65F5F', '#E6AC27', '#8B4B8B']

# Left: MRR delta (recip - forward) across K_R
ax = axes[0]
x  = np.arange(len(K_RECIP_VALUES))
w  = 0.8 / len(model_order_recip)
for i, m in enumerate(model_order_recip):
    sub    = actual_recip[actual_recip['model'] == m].sort_values('K_R')
    deltas = sub['MRR_delta'].tolist()
    ax.bar(x + i * w, deltas, w, label=m, color=colors[i % len(colors)])
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xticks(x + w * (len(model_order_recip) - 1) / 2)
ax.set_xticklabels([f'K_R={k}' for k in K_RECIP_VALUES])
ax.set_ylabel('MRR Δ  (recip − forward)')
ax.set_title('MRR change from reciprocal filter')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Centre: gold survival rate across K_R
ax2 = axes[1]
for i, m in enumerate(model_order_recip):
    sub      = actual_recip[actual_recip['model'] == m].sort_values('K_R')
    survival = sub['gold_survival'].tolist()
    ax2.plot(K_RECIP_VALUES, survival, marker='o', label=m, color=colors[i % len(colors)])
ax2.set_xlabel('K_R')
ax2.set_ylabel('Gold survival rate')
ax2.set_title('Fraction of gold pairs surviving reciprocal filter')
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax2.set_xticks(K_RECIP_VALUES)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# Right: forward R@5 vs reciprocal R@5 at K_R=5 (side-by-side)
ax3    = axes[2]
x3     = np.arange(len(model_order_recip))
w3     = 0.35
sub_k5 = actual_recip[actual_recip['K_R'] == 5].set_index('model')
fwd_r5   = [sub_k5.loc[m, 'fwd_R@5']   for m in model_order_recip]
recip_r5 = [sub_k5.loc[m, 'recip_R@5'] for m in model_order_recip]
ax3.bar(x3,        fwd_r5,   w3, label='Forward R@5',    color='#4878CF', alpha=0.85)
ax3.bar(x3 + w3,   recip_r5, w3, label='Reciprocal R@5', color='#D65F5F', alpha=0.85)
ax3.set_xticks(x3 + w3 / 2)
ax3.set_xticklabels(model_order_recip, rotation=15, ha='right', fontsize=9)
ax3.set_ylabel('R@5')
ax3.set_title('Forward vs reciprocal R@5  (K_R=5)')
ax3.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax3.legend(fontsize=9)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/reciprocal_matching.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/reciprocal_matching.png')


## 4. Results

In [ ]:
results_df = pd.DataFrame(all_results)

# Per-variant tables
for v_name, _, _, desc in VARIANTS:
    sub = (
        results_df[results_df['variant'] == v_name]
        .sort_values('mrr', ascending=False)
        .drop(columns='variant')
        .set_index('model')
    )
    disp = sub.copy()
    for col in disp.columns:
        disp[col] = disp[col].map('{:.1%}'.format)
    print(f'\n── {v_name.upper()} ({desc}) ──')
    print(disp.to_string())

# Model ranking on the actual variant (most relevant)
actual_rank = (
    results_df[results_df['variant'] == 'actual']
    .sort_values('mrr', ascending=False)
    [['model', 'mrr'] + [f'R@{k}' for k in K_VALUES]]
    .reset_index(drop=True)
)
actual_rank.index += 1  # 1-based rank
print('\n── RANKING on ACTUAL variant (real pipeline proxy) ──')
print(actual_rank.to_string())

In [ ]:
# ── Figure 1 & 2: Recall@k and MRR across variants ───────────────────────────
actual_df   = results_df[results_df['variant'] == 'actual']
model_order = actual_df.sort_values('mrr', ascending=False)['model'].tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Embedding Benchmark — {PROCEDURE_ID}', fontsize=13, fontweight='bold')

# Left: Recall@k (actual variant)
ax    = axes[0]
x     = np.arange(len(K_VALUES))
width = 0.8 / len(MODELS)

for i, m in enumerate(model_order):
    row  = actual_df[actual_df['model'] == m].iloc[0]
    vals = [row[f'R@{k}'] for k in K_VALUES]
    ax.bar(x + i * width, vals, width, label=m)

ax.set_xticks(x + width * (len(MODELS) - 1) / 2)
ax.set_xticklabels([f'R@{k}' for k in K_VALUES])
ax.set_ylabel('Recall')
ax.set_title('Recall@k  (actual variant — real pipeline proxy)')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Right: MRR across all three variants
ax2    = axes[1]
x2     = np.arange(len(model_order))
w2     = 0.25
colors = ['#4878CF', '#6ACC65', '#D65F5F']

for i, (v_name, _, _, _) in enumerate(VARIANTS):
    v_df = results_df[results_df['variant'] == v_name]
    mrrs = [v_df[v_df['model'] == m]['mrr'].values[0] for m in model_order]
    ax2.bar(x2 + i * w2, mrrs, w2, label=v_name, color=colors[i])

ax2.set_xticks(x2 + w2)
ax2.set_xticklabels(model_order, rotation=15, ha='right', fontsize=9)
ax2.set_ylabel('MRR')
ax2.set_title('MRR by model and evaluation variant')
ax2.legend(title='variant')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/transformer_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/transformer_benchmark.png')

In [ ]:
# ── Figure 3: Paraphrase vs semantic breakdown ────────────────────────────────
#
# Near-paraphrases inflate metrics via surface-form matching.
# The non-paraphrase subset is the honest measure of semantic retrieval quality.

best_model = model_order[0]
qdata_df   = pd.DataFrame(per_query_db[(best_model, 'actual')])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f'Paraphrase Analysis — {best_model} (actual variant)', fontsize=12, fontweight='bold')

# Left: Recall@k split by paraphrase flag
ax = axes[0]
x  = np.arange(len(K_VALUES))
w  = 0.35

for offset, flag, label, color in [
    (0, False, f'Semantic  (n={(~qdata_df["is_paraphrase"]).sum()})', '#4878CF'),
    (w, True,  f'Near-para (n={qdata_df["is_paraphrase"].sum()})',   '#D65F5F'),
]:
    sub     = qdata_df[qdata_df['is_paraphrase'] == flag]
    recalls = [(sub['rank'] <= k).mean() if len(sub) else 0.0 for k in K_VALUES]
    ax.bar(x + offset, recalls, w, label=label, color=color)

ax.set_xticks(x + w / 2)
ax.set_xticklabels([f'R@{k}' for k in K_VALUES])
ax.set_ylabel('Recall')
ax.set_title('Recall@k: semantic vs near-paraphrase pairs')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Right: Cosine score distributions
ax2 = axes[1]
ax2.hist(qdata_df['score_correct'], bins=30, alpha=0.6, label='Correct article', color='#4878CF')
ax2.hist(qdata_df['score_top1'],    bins=30, alpha=0.6, label='Top-1 retrieved', color='#D65F5F')
ax2.set_xlabel('Cosine similarity')
ax2.set_ylabel('Count')
ax2.set_title('Cosine score: correct article vs top-1 retrieved')
ax2.legend(fontsize=8)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/paraphrase_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved -> results/paraphrase_analysis.png')

# Semantic-only table for all models
print('\n── Semantic-only metrics (non-paraphrase pairs, actual variant) ──')
sem_rows = []
for cfg in MODELS:
    m  = cfg['name']
    qd = pd.DataFrame(per_query_db[(m, 'actual')])
    sem = qd[~qd['is_paraphrase']]
    if len(sem) == 0:
        continue
    row = {'model': m, 'n': len(sem), 'MRR': float((1 / sem['rank']).mean())}
    for k in K_VALUES:
        row[f'R@{k}'] = float((sem['rank'] <= k).mean())
    sem_rows.append(row)

sem_df = pd.DataFrame(sem_rows).sort_values('MRR', ascending=False)
print(sem_df.to_string(index=False))

In [ ]:
# ── Save results ──────────────────────────────────────────────────────────────
proc_slug = PROCEDURE_ID.replace('/', '_').replace('(', '').replace(')', '')
out_csv   = Path('results') / f'transformer_benchmark_{proc_slug}.csv'
out_json  = Path('results') / f'transformer_benchmark_{proc_slug}.json'

results_df.to_csv(out_csv, index=False)

export = {
    'procedure_id':    PROCEDURE_ID,
    'n_gold_pairs':    len(gold_pairs),
    'n_articles':      len(art_rows),
    'paraphrase_rate': float(gold_df['is_paraphrase'].mean()),
    'aggregate':       results_df.to_dict(orient='records'),
    'per_query':       {
        f'{m}__{v}': per_query_db[(m, v)]
        for m, v in per_query_db
    },
}
with open(out_json, 'w', encoding='utf-8') as fh:
    json.dump(export, fh, indent=2)

print(f'Saved CSV  -> {out_csv}')
print(f'Saved JSON -> {out_json}')

## 5. Interpretation Guide

### Reading the numbers

| Metric | What it means | Suggested threshold |
|--------|--------------|---------------------|
| **MRR** | Mean reciprocal rank of the correct article.  MRR = 1.0 means always rank-1. | > 0.40 on *actual* |
| **R@1** | Fraction of queries where correct article is the single top result. | > 25% on *actual* |
| **R@5** | Fraction recovered in top-5 (the pool you send to the reciprocal filter). | > 50% on *actual* |

### Variant deltas

- **baseline → actual** gap: how much the model relies on shared article identifiers.
  A large gap (> 20 pp MRR) means the model is taking a lexical shortcut — dangerous
  because lobbying statements predate the proposal and never cite article numbers.

- **actual → strict** gap: how much the article header (number + title) helps retrieval.
  A small gap means the model finds the right article even without the header — ideal.

### Paraphrase check

If near-paraphrase Recall@k >> semantic Recall@k, the model is pattern-matching surface
form rather than understanding.  Report **semantic-only** metrics in the thesis to avoid
over-claiming retrieval quality.

### Choosing a cosine threshold for the pipeline

The score distribution plot (Figure 3, right) shows the gap between `score_correct`
and `score_top1`.  When `score_correct < score_top1` (rank > 1 cases), the overlap
tells you how tight the decision boundary is.  Pick `MIN_SCORE` at the point where
the two histograms diverge — pairs below that threshold are likely noise regardless
of `forward_k` / `reverse_k` settings.

## 6. Qualitative Error Analysis

Manual inspection of best hits, hard failures, and a score-stratified sample for the
best-performing model.  These cases let you judge whether the model's successes and
failures are semantically reasonable — a necessary complement to aggregate metrics.

In [ ]:
# ── Qualitative Error Analysis — best model, actual variant ──────────────────
TRUNC_FB  = 600   # chars shown for feedback text
TRUNC_ART = 400   # chars shown for article text

best_model = model_order[0]
qdata      = per_query_db[(best_model, 'actual')]

# ── Case selector helpers ─────────────────────────────────────────────────────

def _sample_near_percentile(qdata, pct):
    scores = [d['score_correct'] for d in qdata]
    target = float(np.percentile(scores, pct))
    return min(range(len(qdata)), key=lambda i: abs(qdata[i]['score_correct'] - target))


def print_case(label, qi, d):
    row      = gold_df.iloc[qi]
    art_gold = art_rows[gold_art_idxs[qi]]

    sep = '─' * 72
    print(f'\n{sep}')
    print(f'  {label}')
    print(f'  Rank {d["rank"]:>3}  |  score_correct={d["score_correct"]:.4f}  '
          f'score_top1={d["score_top1"]:.4f}  ROUGE-L={d["rouge_l"]:.3f}')
    print(f'  Org: {row["org"] or "—"}')

    print(f'\n  FEEDBACK (citations stripped):')
    print(f'  {row["feedback_clean"][:TRUNC_FB]!r}')

    gold_header = f'{art_gold["element_type"].capitalize()} {art_gold["element_number"]}'
    if art_gold.get('title'):
        gold_header += f': {art_gold["title"]}'
    print(f'\n  GOLD   [{gold_header}]:')
    print(f'  {art_gold["content"][:TRUNC_ART]!r}')

    # top1_art_idx only present if tt-15 was re-run after the notebook update
    if d['rank'] != 1 and d.get('top1_art_idx') is not None:
        art_top1   = art_rows[d['top1_art_idx']]
        top_header = f'{art_top1["element_type"].capitalize()} {art_top1["element_number"]}'
        if art_top1.get('title'):
            top_header += f': {art_top1["title"]}'
        print(f'\n  TOP-1  [{top_header}]  ← wrong retrieval:')
        print(f'  {art_top1["content"][:TRUNC_ART]!r}')
    elif d['rank'] != 1:
        print(f'\n  TOP-1  [re-run tt-15 to see which article was retrieved instead]')

    print(f'{sep}')


# ── 1. Best hits: rank=1, sorted by score_correct desc ───────────────────────
print('=' * 72)
print(f'  BEST HITS — model: {best_model}  (rank=1, highest confidence)')
print('=' * 72)

best_hits = sorted(
    [(i, d) for i, d in enumerate(qdata) if d['rank'] == 1],
    key=lambda x: -x[1]['score_correct'],
)
for qi, d in best_hits[:3]:
    print_case('BEST HIT', qi, d)


# ── 2. Hard failures: rank>10, highest score_correct ─────────────────────────
print('\n' + '=' * 72)
print(f'  HARD FAILURES — rank > 10, highest score for correct article')
print('=' * 72)

hard_fail = sorted(
    [(i, d) for i, d in enumerate(qdata) if d['rank'] > 10],
    key=lambda x: -x[1]['score_correct'],
)
for qi, d in hard_fail[:3]:
    print_case('HARD FAILURE', qi, d)


# ── 3. Score-stratified sample: p10 / p30 / p50 / p70 / p90 ─────────────────
print('\n' + '=' * 72)
print(f'  STRATIFIED SAMPLE — one case per score percentile')
print('=' * 72)

for pct in [10, 30, 50, 70, 90]:
    qi = _sample_near_percentile(qdata, pct)
    d  = qdata[qi]
    print_case(f'p{pct}  (score_correct≈{d["score_correct"]:.4f})', qi, d)

In [ ]:
# ── False-positive inspection: top-5 non-gold retrievals ─────────────────────
#
# For pairs where rank ∈ [2, 5] (model was close but gold wasn't rank-1),
# show what the model retrieved and let us judge whether the gold label
# or the retrieval is the better match.
#
# Two slices:
#   A) Largest score gap  — model is CONFIDENT the gold is wrong
#   B) Smallest score gap — genuine toss-up, citation noise most likely

FP_N        = 5    # cases to show per slice
TRUNC_FB2   = 500
TRUNC_ART2  = 350

best_model = model_order[0]
sim        = sim_db[(best_model, 'actual')]
qdata      = per_query_db[(best_model, 'actual')]

close_miss = [
    (i, d) for i, d in enumerate(qdata)
    if 2 <= d['rank'] <= 5
]

# score gap = score_top1 - score_correct  (positive = model prefers its retrieval)
close_miss_sorted_by_gap = sorted(close_miss, key=lambda x: -(x[1]['score_top1'] - x[1]['score_correct']))

def print_fp_case(label, qi, d):
    row      = gold_df.iloc[qi]
    art_gold = art_rows[gold_art_idxs[qi]]
    scores   = sim[qi]
    ranked   = np.argsort(scores)[::-1]

    gold_header = f'{art_gold["element_type"].capitalize()} {art_gold["element_number"]}'
    if art_gold.get('title'):
        gold_header += f': {art_gold["title"]}'

    sep = '─' * 72
    print(f'\n{sep}')
    print(f'  {label}')
    print(f'  Gold rank: {d["rank"]}  |  score_correct={d["score_correct"]:.4f}  '
          f'score_top1={d["score_top1"]:.4f}  gap={d["score_top1"]-d["score_correct"]:+.4f}')
    print(f'  Org: {row["org"] or "—"}')
    print(f'\n  FEEDBACK:')
    print(f'  {row["feedback_clean"][:TRUNC_FB2]!r}')
    print(f'\n  GOLD [{gold_header}]  (rank {d["rank"]}):')
    print(f'  {art_gold["content"][:TRUNC_ART2]!r}')

    print(f'\n  TOP-5 RETRIEVED:')
    for pos, art_idx in enumerate(ranked[:5], 1):
        a       = art_rows[art_idx]
        ah      = f'{a["element_type"].capitalize()} {a["element_number"]}'
        if a.get('title'):
            ah += f': {a["title"]}'
        marker  = '← GOLD' if art_idx == gold_art_idxs[qi] else ''
        print(f'    #{pos}  [{ah}]  score={scores[art_idx]:.4f}  {marker}')
        print(f'        {a["content"][:200]!r}')
    print(sep)


print('=' * 72)
print(f'  SLICE A — largest score gap (model most confident gold is wrong)')
print(f'  model: {best_model}, actual variant, rank ∈ [2,5]')
print('=' * 72)
for qi, d in close_miss_sorted_by_gap[:FP_N]:
    print_fp_case(f'GAP={d["score_top1"]-d["score_correct"]:+.4f}', qi, d)

print('\n' + '=' * 72)
print(f'  SLICE B — smallest score gap (genuine toss-ups)')
print('=' * 72)
for qi, d in reversed(close_miss_sorted_by_gap[-FP_N:]):
    print_fp_case(f'GAP={d["score_top1"]-d["score_correct"]:+.4f}', qi, d)
